### **Student Information**
GitHub ID: RiyyMiruku

Kaggle name: RiyyMiruku

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

---

## 1. Model Development

**Task and result in one paragraph.** Six-class emotion classification (anger, disgust, fear, joy, sadness, surprise) on 3,360 labelled posts (joy 875, anger 674, sadness 628, disgust 454, surprise 441, fear 288) and 1,440 test posts. The texts read like Reddit comments and news snippets rather than tweets. My final submission averages two emotion-pretrained RoBERTa models (5 seeds each) and copies the training label onto test texts that appear verbatim in train: **public 0.6455 (16th of 101), private 0.6605 (10th, about the top 10%)**. The public ranking moved up six places on the private board, and every submission kept the same order on both boards.

**How I evaluated.** I first used a fixed stratified 15% holdout (504 posts). It turned out to be too noisy: across 5 folds the same model scored anywhere from 0.606 to 0.660, a spread of about 0.05. From the middle of the competition on, every decision was made on **5-fold out-of-fold (OOF) predictions over all 3,360 posts**, and the public leaderboard (a subset of the test set) was only a sanity check.

### 1.1 Preprocessing Steps

* **Joining the files.** `final_posts.json` (text), `data_identification.csv` (train/test split) and `emotion.csv` (labels) are merged on `post_id`.
* **Data-quality checks.** 50 training texts occur more than once, and 16 of them carry **different labels** for the same text; 59 test texts also appear word for word in the training set. The labels themselves are noisy: posts like *"This book was heartbreaking!"* are labelled fear.
* **Minimal cleaning on purpose.** For the transformers I only trim whitespace. I keep case, punctuation (e.g. "!"), emoji and swear words, because they are emotion cues and the models' own tokenizers handle them. The TF-IDF baseline only lowercases.
* **Cleaning the labels — tried, not adopted.** I used confident learning on the OOF predictions to flag posts whose given label had low probability while another class was confident (strict: 154 posts = 4.6%; loose: 500 = 14.9%). About half of the flagged posts looked like odd labels, the other half like real model mistakes (sarcasm, slang, fragments without context). Retraining without them, evaluated on the *uncleaned* holdout, peaked earlier (epoch 2) but ended no better (0.639 / 0.645 vs 0.644), and picking epoch 2 would have meant tuning on the holdout. My conclusion: part of the "noise" is a systematic labelling convention that the test set very likely shares, so I kept all training data.

### 1.2 Feature Engineering Steps

* **Baseline features.** Word 1–2 gram TF-IDF plus character (`char_wb`) 2–5 gram TF-IDF, sublinear term frequency, fitted on train + test text (labels never used): 81,823 features.
* **Main features: pre-trained representations.** Each transformer uses its own subword tokenizer with `max_length = 128`; the median post is 19 tokens and only 0.8% of posts are longer than 128, so truncation loses almost nothing. I did not hand-craft features: the pre-trained encoders are the feature extractors, and the experiments below showed that *which pre-training* they had matters far more than anything I could add.
* **How much the encoder has to adapt.** With the encoder frozen and a logistic regression on its sentence vectors, OOF macro F1 was only 0.546 (emotion RoBERTa) and 0.518 (generic RoBERTa), against 0.627 with full fine-tuning — updating the encoder is worth about +0.08 here.

### 1.3 Explanation of Your Model

**Final model.** Two emotion-pretrained encoders, each fully fine-tuned with a new 6-class head (AdamW, weight decay 0.01, batch 16, 4 epochs, 10% linear warm-up, gradient clipping 1.0, bf16, on an RTX 3060):

| Component | Pre-training | lr | OOF macro F1 |
|---|---|---|---|
| **HL** `j-hartmann/emotion-english-roberta-large` | 6 emotion datasets, 7 classes (our 6 + neutral) | 1e-5 | 0.6272 |
| **goemo** `SamLowe/roberta-base-go_emotions` | GoEmotions (Reddit, 28 emotions) | 2e-5 | 0.6224 |
| **0.5 × HL + 0.5 × goemo** | | | **0.6413** (better in 4 of 5 folds) |

For the submission each model is trained on all 3,360 posts with 5 seeds (42, 1, 2, 3, 4) and the ten probability vectors are averaged 50/50. Seeds disagree on about 8.5% of the test posts, so averaging them reduces variance. Finally, the 59 test texts that appear verbatim in train take their training label (the model already agreed on 56, so 3 predictions change).

**Why this combination.** Earlier I mixed HL with models from the *same* family (its distilled version, generic RoBERTa) and never got past 0.644 on the holdout. goemo is the first partner that helped: it is about as strong as HL but learned from a different source (Reddit), so it makes different mistakes and the average cancels some of them.

**What I tried and dropped**

| Experiment | Result | Why dropped |
|---|---|---|
| TF-IDF + LogisticRegression (C=5, balanced) | 5-fold 0.502; public 0.5149 / private 0.5173 | baseline only |
| generic `roberta-large` | holdout 0.618 | no emotion prior; 0.026 below HL |
| `emotion-english-distilroberta-base` | holdout 0.632 | fast but weaker than the large version |
| `deberta-v3-large` | loss NaN from epoch 1 in bf16 and fp32 | a single forward/backward step was fine; cause unclear, dropped under time pressure |
| `Qwen3.5-2B-Base` + LoRA (r=16, α=32) + classification head | holdout 0.522 / 0.561 / 0.557 per epoch | gradient checkpointing cut VRAM from 17.8 GB to 5.5 GB, but a 2B general LLM on 3k noisy posts loses to a 355M emotion encoder |
| class-weight tuning on half the holdout | tuned half 0.688 → 0.699, other half 0.601 → 0.593 | overfits the noise |
| head initialised from j-hartmann's 7-class layer (neutral dropped) | holdout 0.640 vs 0.644 | faster start (epoch 1: 0.646 vs 0.637), same end point; zero-shot alone already gets 0.611 |
| learnable 6×6 noise-transition layer | holdout 0.638 vs 0.644 | the learned matrix stayed at its initial diagonal (~0.91), so it found no structure to absorb; the test labels carry the same noise anyway |
| stacking (LR on OOF log-probs + margin + entropy) | OOF 0.6427 vs 0.6413 | gain within fold noise |
| mean pooling + multi-sample dropout head | OOF 0.6223 vs 0.6272 | worse alone, +0.002 in the ensemble |
| irony model `twitter-roberta-base-irony` as a stacking feature | OOF 0.632–0.637 | it flags 31% of posts as ironic (mean 0.37): a tweet model over-fires on this mixed text |
| same irony model fine-tuned as a third ensemble member | OOF 0.6466 (+0.005), submission #6: public 0.6356, private 0.6503 | gain right at my noise threshold with one fold negative; I kept #5, which also won on private (0.6605) |

**Error analysis.** HL's weakest class is disgust (holdout F1 0.49, 20 of 68 disgust posts predicted as anger); the best is joy (0.79). Negative posts that contain positive or ironic cue words ("great", "lol", 🤣) are wrong 50.6% of the time against 38.4% for other negative posts, so the model does miss sarcasm, but by a keyword estimate these posts are only about 2.4% of the data. Most remaining errors are Pidgin English, fragments without context, or labels that two independent judges (an LLM and the fine-tuned model) both disagree with.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

I used Claude (Claude Code) as a coding assistant throughout the competition. My main prompts, what it suggested, and what I decided:

| My prompt (summarised, originally in Chinese) | AI suggestion | Decision and why |
|---|---|---|
| "Read the whole Lab 2 spec, list what we need to do, then focus on Kaggle first." | Submit a quick TF-IDF + LR baseline first, then fine-tune a transformer. | **Adopted** — a safe score above 0.5 before experimenting. |
| (follow-up while setting up) | Use emotion-pretrained checkpoints (j-hartmann) instead of a generic BERT, because their label set almost matches ours. | **Adopted** — +0.026 over generic RoBERTa-large on the holdout. |
| "Is there a rule on which models we can use? Can we use a small pre-trained LLM? I want to try LoRA — check Hugging Face for Qwen 3.5." | Checked the rules (any model allowed, open licence, reproducible); set up Qwen3.5-2B-Base with LoRA and a classification head. | **Rejected** after testing — 0.557 vs 0.644. |
| "Look at the errors of our best model — where can the pipeline improve?" | Error analysis; per-class weight tuning; reuse the pretrained head. | Weight tuning **rejected** (overfits noise); head init **rejected** (no gain). |
| "Can we manually remove the mislabeled training rows and test it?" | Do it automatically (confident learning on OOF), evaluate on an uncleaned holdout. | **Rejected** — no stable gain; the test set likely shares the labelling convention. |
| "Can parallel agents label the data, first train then test, as a local ground truth?" | Only label a train sample; labelling test rows conflicts with rule 4b and cannot be reproduced. | **Partly** — 300 train posts with 4 cheap LLM agents: 57.3% agreement, so LLM labels are not a usable ground truth. |
| "Can we add another head to absorb the noise we cannot explain?" | Noise-adaptation (forward-correction) layer. | **Rejected** — no gain. |
| "Keep exploring better methods." | Add a second emotion model with a different pre-training source (GoEmotions, Reddit) and verify on 5-fold OOF. | **Adopted** — OOF +0.014, public +0.031; this is my final model. |
| "Test stacking first, then a mean-pooling / multi-sample-dropout head, all on 5-fold OOF." | Implemented both with nested CV. | **Rejected** — gains within noise. |
| "Does our model miss sarcasm? Can we fine-tune the irony model too?" | Irony probability as a stacking feature; then fine-tune the irony model as a third member. | **Rejected** — the feature hurt; the 3-model ensemble was +0.005 on OOF but lower on public, and also lower on private. |

**What I took from it.** The AI was most useful for setting up experiments quickly and for keeping me honest about evaluation (moving from a 504-post holdout to 5-fold OOF, and not tuning on the test set). Most of its and my ideas did not survive testing. The one that did — combining two emotion models with different pre-training — fits the main lesson of this competition: on a small, noisy dataset, the knowledge a model brings from pre-training matters more than clever architecture or label cleaning.

---


**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [1]:
# Load the competition files and join them on post_id
# DATA_DIR holds the Kaggle files (final_posts.json, data_identification.csv, emotion.csv); they are not
# redistributed in this repository. WORK_DIR holds the class probabilities saved by the GPU training runs.
import json, os, re, time
import numpy as np
import pandas as pd

DATA_DIR = "../kaggle_data/"
WORK_DIR = "../kaggle_work/"
LABELS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

posts = pd.DataFrame(json.load(open(DATA_DIR + "final_posts.json", encoding="utf-8")))
df = posts.merge(pd.read_csv(DATA_DIR + "data_identification.csv"), on="post_id")
train = df[df.set == "train"].merge(pd.read_csv(DATA_DIR + "emotion.csv"), on="post_id").reset_index(drop=True)
test = df[df.set == "test"].reset_index(drop=True)
print("train:", train.shape, "| test:", test.shape)
print("label counts:", train.label.value_counts().to_dict())

# Data-quality checks: repeated texts, conflicting labels, texts shared by train and test
dup_groups = train.groupby("text").label.agg(["size", "nunique"])
print("repeated train texts:", (dup_groups["size"] > 1).sum(), "| with conflicting labels:", (dup_groups["nunique"] > 1).sum())
shared = set(train.text) & set(test.text)
print("test texts that also appear in train:", test.text.isin(shared).sum())
words = train.text.str.split().str.len()
print(f"words per post: median {words.median():.0f}, max {words.max()}")

# Minimal cleaning: only trim whitespace. Case, punctuation, emoji and swear words are kept on purpose,
# because they carry emotion and the transformer tokenizers handle them natively.
train["text"] = train.text.str.strip()
test["text"] = test.text.str.strip()

# Encode the labels as integers in a fixed order
train["y"] = train.label.map(LABELS.index)


train: (3360, 4) | test: (1440, 3)
label counts: {'joy': 875, 'anger': 674, 'sadness': 628, 'disgust': 454, 'surprise': 441, 'fear': 288}
repeated train texts: 50 | with conflicting labels: 16
test texts that also appear in train: 59
words per post: median 13, max 508


## 2. Feature Engineering Steps

In [2]:
# Baseline features: word 1-2 grams + char_wb 2-5 grams TF-IDF (fitted on train+test text, labels unused)
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer

word_vec = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), sublinear_tf=True)
char_vec = TfidfVectorizer(lowercase=True, analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)
all_text = pd.concat([train.text, test.text])
word_vec.fit(all_text); char_vec.fit(all_text)
X_train_tfidf = hstack([word_vec.transform(train.text), char_vec.transform(train.text)]).tocsr()
X_test_tfidf = hstack([word_vec.transform(test.text), char_vec.transform(test.text)]).tocsr()
print("TF-IDF features:", X_train_tfidf.shape[1])

# Main features: each pre-trained model's own subword tokenizer; check that max_length=128 truncates almost nothing
from transformers import AutoTokenizer

MODELS = {"HL": "j-hartmann/emotion-english-roberta-large",   # RoBERTa-large fine-tuned on 6 emotion datasets
          "goemo": "SamLowe/roberta-base-go_emotions"}        # RoBERTa-base fine-tuned on GoEmotions (Reddit)
MAX_LEN = 128
for tag, name in MODELS.items():
    lengths = AutoTokenizer.from_pretrained(name)(list(train.text))["input_ids"]
    lengths = np.array([len(x) for x in lengths])
    print(f"{tag}: median {np.median(lengths):.0f} tokens, {np.mean(lengths > MAX_LEN):.2%} of posts longer than {MAX_LEN}")


TF-IDF features: 81823


HL: median 19 tokens, 0.80% of posts longer than 128


goemo: median 19 tokens, 0.80% of posts longer than 128


## 3. Model Implementation Steps

In [3]:
# Baseline: LogisticRegression on the TF-IDF features, 5-fold cross-validated macro F1
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score

baseline = LogisticRegression(C=5, max_iter=3000, class_weight="balanced")
pred = cross_val_predict(baseline, X_train_tfidf, train.label, cv=StratifiedKFold(5, shuffle=True, random_state=42))
print(f"TF-IDF + LogisticRegression, 5-fold macro F1: {f1_score(train.label, pred, average='macro'):.4f}")

# Fine-tuning routine used for every transformer run (full fine-tuning, new 6-class head, run on an RTX 3060)
def finetune(model_name, train_idx, predict_texts, seed, epochs=4, lr=1e-5, bs=16):
    import torch
    from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    torch.manual_seed(seed); np.random.seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=6, ignore_mismatched_sizes=True).to("cuda")
    enc = lambda t: {k: v.to("cuda") for k, v in tok(list(t), truncation=True, max_length=MAX_LEN,
                                                    padding=True, return_tensors="pt").items()}
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * ((len(train_idx) + bs - 1) // bs)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    texts, ys = train.text.values, train.y.values
    for _ in range(epochs):
        model.train()
        perm = np.random.permutation(train_idx)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            with torch.autocast("cuda", dtype=torch.bfloat16):
                logits = model(**enc(texts[b])).logits.float()
            loss = torch.nn.functional.cross_entropy(logits, torch.tensor(ys[b], device="cuda"))
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(predict_texts), 64):
            with torch.autocast("cuda", dtype=torch.bfloat16):
                out.append(torch.softmax(model(**enc(predict_texts[i:i + 64])).logits.float(), -1).cpu().numpy())
    return np.concatenate(out)

# Hyper-parameters per model, and the switch between re-training (needs a GPU) and loading saved probabilities
SETTINGS = {"HL": dict(lr=1e-5), "goemo": dict(lr=2e-5)}
SAVED = {"HL": ("oof", "hartmann_large_full"), "goemo": ("goemo_oof", "goemo_full")}
SEEDS = [42, 1, 2, 3, 4]
RUN_TRAINING = False
folds = list(StratifiedKFold(5, shuffle=True, random_state=1).split(train.text, train.y))

# Model selection on 5-fold out-of-fold predictions over all 3,360 labelled posts
oof = {}
for tag, name in MODELS.items():
    oof[tag] = np.zeros((len(train), 6))
    for k, (tr_idx, va_idx) in enumerate(folds):
        if RUN_TRAINING:
            oof[tag][va_idx] = finetune(name, tr_idx, train.text.values[va_idx], seed=42, **SETTINGS[tag])
        else:
            saved_idx = np.load(f"{WORK_DIR}probs/{SAVED[tag][0]}_f{k}_validx.npy")
            oof[tag][saved_idx] = np.load(f"{WORK_DIR}probs/{SAVED[tag][0]}_f{k}_val.npy")
oof["HL + goemo"] = 0.5 * oof["HL"] + 0.5 * oof["goemo"]
for tag, p in oof.items():
    print(f"OOF macro F1 {tag:11s}: {f1_score(train.y, p.argmax(1), average='macro'):.4f}")

# Final model: each network trained on all 3,360 posts with 5 seeds; average the 10 probability sets 50/50
test_probs = {}
for tag, name in MODELS.items():
    runs = []
    for s in SEEDS:
        if RUN_TRAINING:
            runs.append(finetune(name, np.arange(len(train)), test.text.values, seed=s, **SETTINGS[tag]))
        else:
            runs.append(np.load(f"{WORK_DIR}probs/{SAVED[tag][1]}_s{s}_test.npy"))
    test_probs[tag] = np.mean(runs, axis=0)
final = 0.5 * test_probs["HL"] + 0.5 * test_probs["goemo"]
pred = np.array([LABELS[i] for i in final.argmax(1)])

# Post-processing: a test text that appears verbatim in train takes its (majority) train label
train_label = train.groupby("text").label.agg(lambda s: s.mode()[0])
seen = test.text.isin(train_label.index).values
print("duplicate override changes", (pred[seen] != test.text[seen].map(train_label).values).sum(), "of", seen.sum(), "predictions")
pred[seen] = test.text[seen].map(train_label).values

# Write the submission (the scorer expects the columns id,label)
submission = pd.DataFrame({"id": test.post_id, "label": pred})
submission.to_csv("submission.csv", index=False)
print(submission.shape, submission.label.value_counts().to_dict())


TF-IDF + LogisticRegression, 5-fold macro F1: 0.5022
OOF macro F1 HL         : 0.6272
OOF macro F1 goemo      : 0.6224
OOF macro F1 HL + goemo : 0.6413


duplicate override changes 3 of 59 predictions
(1440, 2) {'joy': 395, 'anger': 333, 'sadness': 258, 'surprise': 200, 'disgust': 138, 'fear': 116}
